# Part 0 — Data exploration & building `omics.pkl`

In Session 2, the data was already aligned before you opened a notebook. For this session, this notebook will do that alignment itself. In this notebook, we will inspect three separately-exported omics files, work out exactly where they disagree, and build the single aligned table that every later notebook in this session depends on.

From Part 1 onward, every tool the LLM agent calls is built on one assumption: that this table already exists, in exactly this shape. This notebook is what makes that assumption true.

The function every later notebook relies on expects one object with a strict shape:

```python
load_omics_data(data_dir)  # reads  data_dir / "omics.pkl"
```

> A dict with keys `'transcriptomics'`, `'proteomics'`, `'methylation'`, and `'meta'`
> (the subtype labels), all indexed by the same patients, in the same order.

What is actually sitting in `data/` right now does not meet that bar: three separate files, covering different, only partially-overlapping patients, in different orders. Below, we fix that.

Run this notebook with the `eccb` kernel.

## 0. Setup

This sets where the data lives, and which omics layers and label column we use below.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# claude_notebooks/ sits one level deeper than notebooks/ (notebooks/claude_notebooks/),
# so it needs to climb two levels instead of one to reach the project root.
if Path.cwd().name == "claude_notebooks":
    PROJECT_ROOT = Path.cwd().parent.parent
elif Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"

# The three omics layers used in this session, and the clinical column holding the PAM50 subtype
OMICS = ["transcriptomics", "proteomics", "methylation"]
LABEL_COL = "paper_BRCA_Subtype_PAM50"

print("DATA_DIR:", DATA_DIR)
print("files:", sorted(p.name for p in DATA_DIR.glob("*.pkl")))


DATA_DIR: /home/eidf128/eidf128/shared/export/nfabrega/git/ECCB2026_TEST/sessions/session-3-agentic-llm-workflows/data
files: ['methylation.pkl', 'omics.pkl', 'proteomics.pkl', 'transcriptomics.pkl']


## 1. What's inside each omics file?

Each omics file is a `.pkl` (a pickle — Python's standard way to save an object to disk) holding two tables:

- `expr`: patients × features, the actual measurements
- `meta`: patients × clinical columns, including the PAM50 subtype

Before touching alignment, we need to know what shape we are working with.

In [2]:
raw = {name: pd.read_pickle(DATA_DIR / f"{name}.pkl") for name in OMICS}

# One row per omics layer: how many patients and features it actually has
summary = pd.DataFrame({
    name: {"n_patients": raw[name]["expr"].shape[0],
           "n_features": raw[name]["expr"].shape[1]}
    for name in OMICS
}).T
summary


,n_patients,n_features
transcriptomics,1047,29995
proteomics,869,464
methylation,780,200000


## 2. The omics files are not aligned — reconstructing the shared cohort

The patient counts above already differ across the three files, so they do not share one cohort. First we measure the actual overlap between them, then keep only the patients present in all three, then check that the three files agree on each patient's subtype — independently-exported files can, in principle, disagree.

In [3]:
index_by_omic = {name: raw[name]["expr"].index for name in OMICS}

for name in OMICS:
    print(f"{name:15s}: {len(index_by_omic[name])} patients")

t, p, m = (set(index_by_omic[o]) for o in OMICS)
print("\noverlap - transcriptomics & proteomics :", len(t & p))
print("overlap - transcriptomics & methylation:", len(t & m))
print("overlap - proteomics & methylation     :", len(p & m))
print("overlap - all three                    :", len(t & p & m))


transcriptomics: 1047 patients
proteomics     : 869 patients
methylation    : 780 patients

overlap - transcriptomics & proteomics : 840
overlap - transcriptomics & methylation: 745
overlap - proteomics & methylation     : 631
overlap - all three                    : 603


In [4]:
# Keep only patients measured in every omics layer
common = sorted(set.intersection(*(set(index_by_omic[o]) for o in OMICS)))

# Each patient's subtype label, as recorded independently in each of the three files
labels_wide = pd.DataFrame(
    {name: raw[name]["meta"].loc[common, LABEL_COL] for name in OMICS}
)

# A patient must not be called LumA in one file and Basal in another
consistent = labels_wide.nunique(axis=1, dropna=True).le(1)
assert consistent.all(), "Subtype labels disagree across omics layers for some patients."

subtype = labels_wide[OMICS[0]]
labeled = subtype.dropna().index.tolist()
print(f"patients in all three omics layers : {len(common)}")
print(f"labels consistent across layers    : {int(consistent.sum())}/{len(labels_wide)}")
print(f"patients with a PAM50 label        : {len(labeled)}")


patients in all three omics layers : 603
labels consistent across layers    : 603/603
patients with a PAM50 label        : 603


## 3. Assemble and verify the aligned table

This builds the final table: every omics layer reindexed onto the same patients, in the same order, with the subtype label attached. Then we check it directly against what will be required downstream, so a problem shows up here — not three notebooks from now.

In [5]:
patient_ids = pd.Index(labeled, name="patient_id")

omics = {}
for name in OMICS:
    expr = raw[name]["expr"].loc[patient_ids]
    expr.index = expr.index.astype(str)
    omics[name] = expr

meta = subtype.loc[patient_ids].astype(str)
meta.index = meta.index.astype(str)
omics["meta"] = meta

for k, v in omics.items():
    print(f"{k:15s}: {v.shape}")


transcriptomics: (603, 29995)
proteomics     : (603, 464)
methylation    : (603, 200000)
meta           : (603,)


In [6]:
# Every omics table must share the same patient index, in the same order, with no fully-empty layer
ref_index = omics["meta"].index.astype(str)
for name in OMICS:
    assert omics[name].index.astype(str).equals(ref_index), f"index mismatch in {name}"
    assert not omics[name].isna().all(axis=None), f"{name} is entirely NaN"
print("All checks passed.")


All checks passed.


## 4. Save `omics.pkl`

We write the table to disk, then read it back through `load_omics_data` — the function every later notebook in this session calls — to confirm it is not just internally consistent, but exactly what the tools expect.

In [7]:
out_path = DATA_DIR / "omics.pkl"
pd.to_pickle(omics, out_path)
print("wrote:", out_path, f"({out_path.stat().st_size / 1e6:.1f} MB)")

# Round-trip check via the actual function every later notebook calls.
import sys
sys.path.insert(0, str(PROJECT_ROOT))
from src.mofa_tools import load_omics_data

X_omics, y = load_omics_data(DATA_DIR)
print("load_omics_data OK:", {k: v.shape for k, v in X_omics.items()}, "| y:", y.shape)


wrote: /home/eidf128/eidf128/shared/export/nfabrega/git/ECCB2026_TEST/sessions/session-3-agentic-llm-workflows/data/omics.pkl (1115.0 MB)
load_omics_data OK: {'transcriptomics': (603, 29995), 'proteomics': (603, 464), 'methylation': (603, 200000)} | y: (603,)


## Bridge to Part 1

With `omics.pkl` on disk and `load_omics_data` verified to read it back cleanly, every notebook from here on can take the data layer for granted. Part 1 picks up exactly here: the same `mofa_tools.py` functions get wrapped as tools an LLM can call.